In [ ]:
#d2025

In [ ]:
import sqlite3
import pandas as pd
import os

mesh_file_path = r"C:\Users\23062\Desktop\d2025.bin"
db_path = r"F:\Pycharm_Temp\S.db"
# =======================================

def parse_official_mesh_file():
    print(f"=== 正在解析官方 MeSH 文件: {os.path.basename(mesh_file_path)} ===")

    data = []

    current_ui = None
    tree_numbers = []

    try:
        with open(mesh_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if line == '*NEWRECORD':
                    # 保存上一条记录
                    if current_ui and tree_numbers:
                        trees_str = " ".join(tree_numbers)
                        data.append((current_ui, trees_str))

                    # 重置
                    current_ui = None
                    tree_numbers = []
                    continue

                # 提取 UI (Unique Identifier)
                if line.startswith('UI = '):
                    current_ui = line.split(' = ')[1].strip()

                # 提取 MN (MeSH Tree Number)
                elif line.startswith('MN = '):
                    mn = line.split(' = ')[1].strip()
                    tree_numbers.append(mn)

            # 保存最后一条
            if current_ui and tree_numbers:
                trees_str = " ".join(tree_numbers)
                data.append((current_ui, trees_str))

    except FileNotFoundError:
        print("错误：找不到文件，请检查路径。")
        return
    except Exception as e:
        print(f"解析出错: {e}")
        return

    print(f"--> 解析完成！共提取到 {len(data)} 个 MeSH 词条。")

    if len(data) > 10000:
        print("Step 2: 存入数据库 mesh_tree_map_official ...")
        conn = sqlite3.connect(db_path)
        df = pd.DataFrame(data, columns=['ui', 'tree_numbers'])

        # 存入一张新表，以此区分旧表
        df.to_sql('mesh_tree_map_official', conn, if_exists='replace', index=False)

        # 建索引
        conn.execute("CREATE INDEX IF NOT EXISTS idx_mesh_off_ui ON mesh_tree_map_official (ui)")
        conn.close()
        print("=== 官方字典入库成功！ ===")
        print("提示：后续计算时，请使用表名 'mesh_tree_map_official'")
    else:
        print("警告：提取数量太少，可能文件格式不对。")

if __name__ == '__main__':
    # 记得修改上面的 mesh_file_path 路径！
    parse_official_mesh_file()

In [ ]:
import sqlite3
import pandas as pd
import time
import os

db_path = r"F:\Pycharm_Temp\S.db"
a02_path = r"F:\GG\20251103-132622\PKG2020S4_A02_AuthorList.sql\PKG2020S4_A02_AuthorList.sql"

TARGET_COL_IDX = -4
# ===========================================

def final_import_paper_authors():
    print(f"=== 启动 17.9GB 数据全量导入程序 ===")

    # 1. 准备白名单 (Valid IDs)
    print("Step 1: 加载目标学者 ID (白名单)...")
    conn = sqlite3.connect(db_path)
    try:
        # 只保留 career_length >= 20 的资深学者
        df_ids = pd.read_sql("SELECT scholar_id FROM final_scholars WHERE career_length >= 20 AND scholar_id != 0", conn)
        valid_ids = set(df_ids['scholar_id'].astype(int))
        print(f"--> 白名单加载完成: 需追踪 {len(valid_ids)} 位学者。")
    except Exception as e:
        print(f"数据库错误: {e}")
        return
    finally:
        conn.close()

    # 2. 准备数据库环境 (先删索引，加速写入)
    print("\nStep 2: 初始化数据库环境...")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 建表
    cursor.execute("CREATE TABLE IF NOT EXISTS paper_authors (author_id INTEGER, pmid INTEGER)")
    # 清空旧数据 (防止重复)
    cursor.execute("DELETE FROM paper_authors")
    # !!! 关键操作：删除索引以加速写入 !!!
    print("--> 正在移除旧索引以加速写入...")
    cursor.execute("DROP INDEX IF EXISTS idx_pa_aid")
    cursor.execute("DROP INDEX IF EXISTS idx_pa_pmid")
    conn.commit()

    # 3. 开始流式处理
    file_size = os.path.getsize(a02_path)
    print(f"\nStep 3: 开始扫描文件 (总大小: {file_size / (1024**3):.2f} GB)")
    print("--> 进度将按 [文件读取百分比] 显示，请耐心等待...")

    batch_data = []
    BATCH_SIZE = 100000  # 每10万条写入一次
    total_saved = 0

    start_time = time.time()

    try:
        with open(a02_path, 'r', encoding='utf-8', errors='ignore') as f:
            while True:
                # 逐行读取，但这里的一行可能巨大
                line = f.readline()
                if not line: break

                if not line.startswith("INSERT INTO"): continue

                try:
                    start_pos = line.find('(')
                    if start_pos == -1: continue

                    content = line[start_pos:].strip()
                    if content.endswith(';'): content = content[:-1]

                    tuples = content.split("),(")

                    for t in tuples:
                        clean_t = t.replace('(', '').replace(')', '')
                        parts = clean_t.split(',')

                        try:
                            pmid = int(parts[1].strip())

                            raw_auth = parts[TARGET_COL_IDX].strip().replace("'", "")

                            if raw_auth == "" or raw_auth.upper() == "NULL": continue

                            author_id = int(float(raw_auth))

                            if author_id in valid_ids:
                                batch_data.append((author_id, pmid))

                        except (ValueError, IndexError):
                            continue

                except Exception:
                    continue

                # 批量写入
                if len(batch_data) >= BATCH_SIZE:
                    cursor.executemany("INSERT INTO paper_authors VALUES (?, ?)", batch_data)
                    conn.commit()
                    total_saved += len(batch_data)
                    batch_data = [] # 清空内存

                current_pos = f.tell()
                progress = (current_pos / file_size) * 100

                elapsed = time.time() - start_time
                if elapsed > 0:
                    speed = current_pos / elapsed  # bytes per second
                    remaining_bytes = file_size - current_pos
                    eta = remaining_bytes / speed if speed > 0 else 0

                    if int(progress * 10) % 5 == 0: # 每0.5%打印一次
                        print(f"\r进度: {progress:.2f}% | 已存关联: {total_saved} | ETA: {eta/60:.1f} 分钟", end="")

    except KeyboardInterrupt:
        print("\n\n!!! 用户手动中断，正在保存缓冲区数据...")
    except Exception as e:
        print(f"\n\n!!! 发生未预期的错误: {e}")

    # 保存剩余数据
    if batch_data:
        cursor.executemany("INSERT INTO paper_authors VALUES (?, ?)", batch_data)
        conn.commit()
        total_saved += len(batch_data)

    print(f"\n\n扫描结束！共保存关联记录: {total_saved} 条。")

    # 4. 重建索引
    print("\nStep 4: 正在重建索引 (请勿关闭，这步完成后才算真正结束)...")
    start_idx = time.time()
    cursor.execute("CREATE INDEX idx_pa_aid ON paper_authors (author_id)")
    cursor.execute("CREATE INDEX idx_pa_pmid ON paper_authors (pmid)")
    conn.commit()
    print(f"--> 索引重建完成，耗时 {(time.time() - start_idx)/60:.1f} 分钟。")

    conn.close()
    print(f"=== 全部完成！现在 paper_authors 表已就绪 ===")
    print("下一步：请运行‘生成生涯演化矩阵’代码。")

if __name__ == '__main__':
    final_import_paper_authors()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import time

db_path = r"F:\Pycharm_Temp\S.db"

def generate_evolution_matrix_final():
    print("=== 开始构建生涯演化矩阵 (Final Version) ===")
    conn = sqlite3.connect(db_path)

    # 1. 加载 MeSH 官方字典 (UI -> Tree Numbers)
    print("Step 1: 加载 MeSH 官方字典...")
    try:
        df_map = pd.read_sql("SELECT ui, tree_numbers FROM mesh_tree_map_official", conn)
        # 构造字典: D000001 -> "A01.111 B02.222"
        mesh_dict = dict(zip(df_map['ui'], df_map['tree_numbers']))
        print(f"--> 字典加载完成: {len(mesh_dict)} 条")
    except Exception as e:
        print(f"字典加载失败，请检查表名是否正确: {e}")
        return

    # 2. 获取资深学者名单
    print("Step 2: 获取资深学者基础信息...")
    try:
        df_scholars = pd.read_sql("SELECT scholar_id, first_pub_year FROM final_scholars WHERE career_length >= 20 AND scholar_id != 0", conn)
        target_ids = set(df_scholars['scholar_id'])
        # 字典: 用于快速计算职业年份 (PubYear - FirstYear + 1)
        scholar_start_map = dict(zip(df_scholars['scholar_id'], df_scholars['first_pub_year']))
        print(f"--> 目标学者数量: {len(target_ids)}")
    except Exception as e:
        print(f"学者名单读取失败: {e}")
        return

    # 3. 流式读取并计算
    print("Step 3: 流式处理文章数据 (这步最慢，请耐心等待)...")

    sql = """
    SELECT
        pa.author_id,
        a.pub_year,
        mh.descriptor_name_ui,
        IFNULL(pc.citation_count, 0) as cites
    FROM paper_authors pa
    JOIN articles a ON pa.pmid = a.pmid
    JOIN mesh_headings mh ON pa.pmid = mh.pmid
    LEFT JOIN pmid_citations pc ON pa.pmid = pc.pmid
    WHERE pa.author_id IN (SELECT scholar_id FROM final_scholars WHERE career_length >= 20 AND scholar_id != 0)
    """

    chunk_size = 200000
    reader = pd.read_sql(sql, conn, chunksize=chunk_size)

    scholar_history = {}

    start_time = time.time()
    count = 0

    for chunk in reader:
        count += 1
        if count % 10 == 0:
            elapsed = time.time() - start_time
            print(f"\r已处理 {count * chunk_size // 10000} 万行数据 | 耗时: {elapsed/60:.1f} 分钟", end="")

        # 3.1 映射 Tree Number
        chunk['tree_str'] = chunk['descriptor_name_ui'].map(mesh_dict)
        chunk = chunk.dropna(subset=['tree_str'])

        if chunk.empty: continue

        # 3.2 解析深度和广度 (Vectorized operation is hard here, using apply)
        def parse_metrics(t_str):
            trees = t_str.split() # 按空格分割多个位置
            # 深度：取最大深度
            d = max([t.count('.') + 1 for t in trees])
            # 广度：返回所有首字母的列表
            cats = [t[0] for t in trees]
            return d, cats

        # 应用解析
        parsed = chunk['tree_str'].apply(parse_metrics)
        chunk['depth'] = [x[0] for x in parsed]
        chunk['cats'] = [x[1] for x in parsed]

        # 3.3 聚合到 Python 字典中

        arr_auth = chunk['author_id'].values
        arr_year = chunk['pub_year'].values
        arr_depth = chunk['depth'].values
        arr_cats = chunk['cats'].values
        arr_cites = chunk['cites'].values

        for i in range(len(chunk)):
            sid = arr_auth[i]
            pub_year = arr_year[i]

            start_year = scholar_start_map.get(sid)
            if not start_year: continue

            # 计算职业年龄
            c_age = int(pub_year - start_year + 1)

            if 1 <= c_age <= 20:
                if sid not in scholar_history:
                    scholar_history[sid] = {}

                if c_age not in scholar_history[sid]:
                    scholar_history[sid][c_age] = {
                        'depth_sum': 0, 'depth_cnt': 0,
                        'cats_set': set(),
                        'cite_sum': 0, 'cite_cnt': 0
                    }

                # 累加深度
                scholar_history[sid][c_age]['depth_sum'] += arr_depth[i]
                scholar_history[sid][c_age]['depth_cnt'] += 1

                # 累加广度 (集合去重)
                scholar_history[sid][c_age]['cats_set'].update(arr_cats[i])

                scholar_history[sid][c_age]['cite_sum'] += arr_cites[i]
                scholar_history[sid][c_age]['cite_cnt'] += 1

    print("\nStep 4: 汇总并生成最终表格...")

    final_records = []

    for sid, history in scholar_history.items():
        if len(history) < 3:
            continue

        record = {'scholar_id': sid}

        # 填充 1-20 年
        for year in range(1, 21):
            if year in history:
                data = history[year]

                # 年度平均深度
                avg_depth = data['depth_sum'] / data['depth_cnt'] if data['depth_cnt'] > 0 else np.nan

                # 年度广度 (涉及的一级学科数量)
                breadth = len(data['cats_set'])

                # 年度平均引用
                avg_cite = data['cite_sum'] / data['cite_cnt'] if data['cite_cnt'] > 0 else 0

                record[f'd_{year}'] = avg_depth
                record[f'b_{year}'] = breadth
                record[f'c_{year}'] = avg_cite
            else:
                record[f'd_{year}'] = np.nan
                record[f'b_{year}'] = np.nan
                record[f'c_{year}'] = np.nan # 引用缺失填NaN，后面处理

        final_records.append(record)

    # 存入 DataFrame
    df_final = pd.DataFrame(final_records)
    print(f"--> 原始记录数: {len(df_final)}")

    # 5. 数据插值与清洗 (Interpolation)
    print("Step 5: 数据插值处理 (填补空白年份)...")

    cols_d = [f'd_{i}' for i in range(1, 21)]
    cols_b = [f'b_{i}' for i in range(1, 21)]
    cols_c = [f'c_{i}' for i in range(1, 21)]

    # 线性插值填补中间的空缺 (比如第3年没发文章，取2和4的平均)
    df_final[cols_d] = df_final[cols_d].interpolate(axis=1, limit_direction='both')
    df_final[cols_b] = df_final[cols_b].interpolate(axis=1, limit_direction='both')
    df_final[cols_c] = df_final[cols_c].fillna(0)

    # 存入数据库
    print("Step 6: 保存到表 scholar_evolution_matrix ...")
    df_final.to_sql('scholar_evolution_matrix', conn, if_exists='replace', index=False)

    conn.close()
    print(f"=== 矩阵构建完成！共 {len(df_final)} 位合格学者。 ===")
    print("下一步：执行 K-Means 聚类分析。")

if __name__ == '__main__':
    generate_evolution_matrix_final()

In [ ]:
#开聚

In [ ]:
#手肘

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os

db_path = r"F:\Pycharm_Temp\S.db"
save_dir = r"F:\Pycharm_Temp\True_Project"
if not os.path.exists(save_dir): os.makedirs(save_dir)

COLORS = {
    'line':   '#4477AA',
    'point':  '#EE6677',
    'text':   '#333333',
    'grid':   '#E5E5E5'
}

def plot_elbow_method_academic_label():
    print("=== 开始生成手肘法图表 (学术标签修正版) ===")
    conn = sqlite3.connect(db_path)

    # 1. 读取数据
    cols_b = [f'b_{i}' for i in range(1, 21)]
    cols_d = [f'd_{i}' for i in range(1, 21)]
    df_matrix = pd.read_sql(f"SELECT scholar_id, {','.join(cols_b)}, {','.join(cols_d)} FROM scholar_evolution_matrix", conn)
    conn.close()

    # 2. 特征工程
    X_features = pd.DataFrame()
    vals_b = df_matrix[cols_b].values
    vals_d = df_matrix[cols_d].values
    X_features['b_mean'] = np.mean(vals_b, axis=1)
    X_features['b_slope'] = (np.mean(vals_b[:, -3:], axis=1) - np.mean(vals_b[:, :3], axis=1))
    X_features['d_mean'] = np.mean(vals_d, axis=1)
    X_features['d_slope'] = (np.mean(vals_d[:, -3:], axis=1) - np.mean(vals_d[:, :3], axis=1))
    X_features['ratio'] = X_features['d_mean'] / (X_features['b_mean'] + 1e-5)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_features)

    # 3. 计算WCSS
    inertias = []
    k_range = range(1, 11)
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_scaled)
        inertias.append(kmeans.inertia_)

    # 4. 绘图
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
    plt.rcParams['text.color'] = COLORS['text']
    plt.rcParams['axes.labelcolor'] = COLORS['text']
    plt.rcParams['xtick.color'] = COLORS['text']
    plt.rcParams['ytick.color'] = COLORS['text']

    sns.set_style("whitegrid", {'grid.color': COLORS['grid']})

    fig, ax = plt.subplots(figsize=(8, 6))

    # 画线
    ax.plot(k_range, inertias, color=COLORS['line'],
            linewidth=2.5, linestyle='-', zorder=2, marker='o', markersize=8,
            label='Within-Cluster Sum of Squares')

    # 高亮 K=4
    elbow_k = 4
    elbow_val = inertias[elbow_k-1]
    ax.plot(elbow_k, elbow_val, marker='*', markersize=22,
            color=COLORS['point'], markeredgecolor='white', markeredgewidth=1.5,
            zorder=3, label=f'Selected Optimal K={elbow_k}', linestyle='None')

    # --- Y轴标签 ---
    ax.set_xlabel('Number of Clusters (K)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=14, fontweight='bold', labelpad=10)

    plt.xticks(k_range, fontsize=12)
    plt.yticks(fontsize=12)

    # 图例
    legend = plt.legend(fontsize=11, loc='upper right', frameon=True)
    legend.get_frame().set_edgecolor('#CCCCCC')
    legend.get_frame().set_facecolor('white')

    sns.despine(top=True, right=True)
    plt.tight_layout()

    save_path = os.path.join(save_dir, "Figure_Elbow_Method_Academic.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"=== 学术修正版手肘图已保存至: {save_path} ===")

if __name__ == '__main__':
    plot_elbow_method_academic_label()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
import os

db_path = r"F:\Pycharm_Temp\S.db"
save_dir = r"F:\Pycharm_Temp\True_Project"
if not os.path.exists(save_dir): os.makedirs(save_dir)

def plot_performance_with_linestyles():
    print("=== 开始生成绩效图  ===")
    conn = sqlite3.connect(db_path)

    # 1. 准备数据与聚类
    cols_b = [f'b_{i}' for i in range(1, 21)]
    cols_d = [f'd_{i}' for i in range(1, 21)]
    cols_c = [f'c_{i}' for i in range(1, 21)]
    df_matrix = pd.read_sql(f"SELECT scholar_id, {','.join(cols_b)}, {','.join(cols_d)}, {','.join(cols_c)} FROM scholar_evolution_matrix", conn)

    X_features = pd.DataFrame()
    vals_b = df_matrix[cols_b].values
    vals_d = df_matrix[cols_d].values
    X_features['b_mean'] = np.mean(vals_b, axis=1)
    X_features['b_slope'] = (np.mean(vals_b[:, -3:], axis=1) - np.mean(vals_b[:, :3], axis=1))
    X_features['d_mean'] = np.mean(vals_d, axis=1)
    X_features['d_slope'] = (np.mean(vals_d[:, -3:], axis=1) - np.mean(vals_d[:, :3], axis=1))
    X_features['ratio'] = X_features['d_mean'] / (X_features['b_mean'] + 1e-5)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_features)
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_matrix['cluster'] = clusters

    cluster_cite_means = df_matrix.groupby('cluster')[cols_c].mean().mean(axis=1)
    sorted_map = {old: new for new, old in enumerate(cluster_cite_means.sort_values(ascending=False).index)}
    df_matrix['cluster_new'] = df_matrix['cluster'].map(sorted_map)
    scholar_clusters = df_matrix[['scholar_id', 'cluster_new']].set_index('scholar_id')['cluster_new'].to_dict()

    # 2. 准备引用数据 (平滑)
    plot_data_cite = []
    for c_id in range(4):
        subset = df_matrix[df_matrix['cluster_new'] == c_id][cols_c]
        raw_means = subset.mean().values
        smooth_means = gaussian_filter1d(raw_means, sigma=1.2)
        # 这里改一下结构，方便直接画图
        for i, val in enumerate(smooth_means):
            plot_data_cite.append({'Cluster': c_id, 'Year': i+1, 'Value': val})
    df_plot_cite = pd.DataFrame(plot_data_cite)

    # 3. 准备生产力数据 (平滑)
    sql_pub = """
    SELECT pa.author_id, a.pub_year, fs.first_pub_year
    FROM paper_authors pa
    JOIN articles a ON pa.pmid = a.pmid
    JOIN final_scholars fs ON pa.author_id = fs.scholar_id
    WHERE fs.career_length >= 20 AND fs.scholar_id != 0
    """
    chunk_size = 500000
    reader = pd.read_sql(sql_pub, conn, chunksize=chunk_size)
    all_chunks = []
    for chunk in reader:
        chunk['career_year'] = chunk['pub_year'] - chunk['first_pub_year'] + 1
        chunk = chunk[(chunk['career_year'] >= 1) & (chunk['career_year'] <= 20)]
        all_chunks.append(chunk[['author_id', 'career_year']])
    df_pubs = pd.concat(all_chunks)
    df_counts = df_pubs.groupby(['author_id', 'career_year']).size().reset_index(name='count')
    df_counts['cluster'] = df_counts['author_id'].map(scholar_clusters)
    df_counts = df_counts.dropna(subset=['cluster'])

    plot_data_pub = []
    for c_id in range(4):
        sub_df = df_counts[df_counts['cluster'] == c_id]
        n_scholars = sum(df_matrix['cluster_new'] == c_id)
        raw_means = []
        for yr in range(1, 21):
            total_pubs = sub_df[sub_df['career_year'] == yr]['count'].sum()
            avg_pub = total_pubs / n_scholars
            raw_means.append(avg_pub)
        smooth_means = gaussian_filter1d(raw_means, sigma=1.2)
        for i, val in enumerate(smooth_means):
            plot_data_pub.append({'Cluster': c_id, 'Year': i+1, 'Value': val})
    df_plot_pub = pd.DataFrame(plot_data_pub)
    conn.close()

    # --- 4. 绘图 (带线型) ---
    print("Step 4: 绘制带线型的组合图...")

    # 设置风格
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
    sns.set_style("whitegrid")

    # 颜色与线型定义
    PALETTE = sns.color_palette("tab10", 4)
    STYLES = {
        0: '-',   # Cluster 0: Solid
        1: '--',  # Cluster 1: Dashed
        2: ':',   # Cluster 2: Dotted (注意，这里为了区分度，停滞型用点线)
        3: '-.'   # Cluster 3: Dash-dot
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --- 左图 ---
    ax1 = axes[0]
    for c_id in range(4):
        subset = df_plot_cite[df_plot_cite['Cluster'] == c_id]
        ax1.plot(subset['Year'], subset['Value'],
                 label=f'Cluster {c_id}',
                 color=PALETTE[c_id],
                 linestyle=STYLES[c_id], # 应用线型
                 linewidth=3.5)          # 加粗

    ax1.set_title('(a) Evolution of Citation Impact', fontsize=16, fontweight='bold', pad=15)
    ax1.set_xlabel('Career Year', fontsize=14)
    ax1.set_ylabel('Avg. Annual Citations', fontsize=14)
    ax1.set_xlim(1, 20)
    ax1.set_xticks(range(2, 21, 2))
    ax1.grid(True, linestyle='-', linewidth=1, color='#d9d9d9', alpha=0.8)
    ax1.set_axisbelow(True)
    ax1.legend(loc='upper left', frameon=True, fontsize=12, title='Scientists Types')
    sns.despine(ax=ax1)

    # --- 右图 ---
    ax2 = axes[1]
    for c_id in range(4):
        subset = df_plot_pub[df_plot_pub['Cluster'] == c_id]
        ax2.plot(subset['Year'], subset['Value'],
                 label=f'Cluster {c_id}',
                 color=PALETTE[c_id],
                 linestyle=STYLES[c_id], # 应用线型
                 linewidth=3.5)

    ax2.set_title('(b) Evolution of Annual Productivity', fontsize=16, fontweight='bold', pad=15)
    ax2.set_xlabel('Career Year', fontsize=14)
    ax2.set_ylabel('Avg. Annual Publications', fontsize=14)
    ax2.set_xlim(1, 20)
    ax2.set_xticks(range(2, 21, 2))
    ax2.grid(True, linestyle='-', linewidth=1, color='#d9d9d9', alpha=0.8)
    ax2.set_axisbelow(True)

    # 右图
    sns.despine(ax=ax2)

    # 调整间距
    plt.subplots_adjust(wspace=0.25, top=0.85, bottom=0.15, left=0.08, right=0.95)

    # 保存
    save_path = os.path.join(save_dir, "Figure_Performance_LineStyles.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"=== 已保存: {save_path} ===")

if __name__ == '__main__':
    plot_performance_with_linestyles()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os

db_path = r"F:\Pycharm_Temp\S.db"
save_dir = r"F:\Pycharm_Temp\True_Project"
if not os.path.exists(save_dir): os.makedirs(save_dir)

def plot_trajectories_with_linestyles():
    print("=== 开始生成带线型区分的轨迹图 (黑白打印友好版) ===")
    conn = sqlite3.connect(db_path)

    # 1. 读取数据
    cols_b = [f'b_{i}' for i in range(1, 21)]
    cols_d = [f'd_{i}' for i in range(1, 21)]
    cols_c = [f'c_{i}' for i in range(1, 21)]

    df_matrix = pd.read_sql(f"SELECT scholar_id, {','.join(cols_b)}, {','.join(cols_d)}, {','.join(cols_c)} FROM scholar_evolution_matrix", conn)
    conn.close()

    # 2. 复现聚类
    X_features = pd.DataFrame()
    vals_b = df_matrix[cols_b].values
    vals_d = df_matrix[cols_d].values
    X_features['b_mean'] = np.mean(vals_b, axis=1)
    X_features['b_slope'] = (np.mean(vals_b[:, -3:], axis=1) - np.mean(vals_b[:, :3], axis=1))
    X_features['d_mean'] = np.mean(vals_d, axis=1)
    X_features['d_slope'] = (np.mean(vals_d[:, -3:], axis=1) - np.mean(vals_d[:, :3], axis=1))
    X_features['ratio'] = X_features['d_mean'] / (X_features['b_mean'] + 1e-5)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_features)
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_matrix['cluster'] = clusters

    # 颜色重映射
    cluster_cite_means = df_matrix.groupby('cluster')[cols_c].mean().mean(axis=1)
    sorted_map = {old: new for new, old in enumerate(cluster_cite_means.sort_values(ascending=False).index)}
    df_matrix['cluster_new'] = df_matrix['cluster'].map(sorted_map)

    # 3. 绘图
    print("Step 2: 绘制包含线型的图表...")

    # 设置风格
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
    sns.set_style("whitegrid")

    # 定义线型字典 (Line Styles)
    STYLES = {
        0: '-',   # Cluster 0: Solid (主角)
        1: '--',  # Cluster 1: Dashed
        2: ':',   # Cluster 2: Dotted (停滞)
        3: '-.'   # Cluster 3: Dash-dot (深耕)
    }

    # 定义颜色
    PALETTE = sns.color_palette("tab10", 4)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    years = range(1, 21)

    # --- 左图: 广度 ---
    ax1 = axes[0]
    for c_id in range(4):
        subset = df_matrix[df_matrix['cluster_new'] == c_id]
        mean_traj = subset[cols_b].mean().values

        ax1.plot(years, mean_traj,
                 label=f'Cluster {c_id}',
                 color=PALETTE[c_id],
                 linestyle=STYLES[c_id],
                 linewidth=3.5)

    ax1.set_title('(a) Evolution of Knowledge Breadth', fontsize=16, fontweight='bold', pad=15)
    ax1.set_xlabel('Career Year', fontsize=14)
    ax1.set_ylabel('Breadth Score', fontsize=14)
    ax1.set_xlim(1, 20)
    ax1.set_xticks(range(2, 21, 2))
    ax1.grid(True, linestyle='-', linewidth=0.5, color='#d9d9d9', alpha=0.8)

    # 图例
    ax1.legend(loc='upper left', fontsize=12, title_fontsize=12, frameon=True)

    # --- 右图: 深度 ---
    ax2 = axes[1]
    for c_id in range(4):
        subset = df_matrix[df_matrix['cluster_new'] == c_id]
        mean_traj = subset[cols_d].mean().values

        ax2.plot(years, mean_traj,
                 label=f'Cluster {c_id}',
                 color=PALETTE[c_id],
                 linestyle=STYLES[c_id], # 关键修改：应用线型
                 linewidth=3.5)

    ax2.set_title('(b) Evolution of Knowledge Depth', fontsize=16, fontweight='bold', pad=15)
    ax2.set_xlabel('Career Year', fontsize=14)
    ax2.set_ylabel('Depth Score', fontsize=14)
    ax2.set_xlim(1, 20)
    ax2.set_xticks(range(2, 21, 2))
    ax2.grid(True, linestyle='-', linewidth=0.5, color='#d9d9d9', alpha=0.8)


    # 调整布局
    plt.tight_layout()
    sns.despine()

    # 保存
    save_path = os.path.join(save_dir, "Figure_Breadth_Depth_LineStyles.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"=== 已保存: {save_path} ===")

if __name__ == '__main__':
    plot_trajectories_with_linestyles()

In [ ]:
#回归

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os

DB_PATH = r"F:\Pycharm_Temp\S.db"
OUTPUT_DIR = r"F:\Pycharm_Temp\True_Project\Appendix"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REFERENCE_CLUSTER = "2"
CLUSTER_ORDER = ["2", "0", "1", "3"]

def prepare_and_run_hierarchical_regressions():
    print("=== 开始执行分层回归分析 (Unadjusted vs Controlled) ===")
    conn = sqlite3.connect(DB_PATH)

    # 1. 准备大宽表数据
    print("Step 1: 整合全量变量数据...")
    df_reg = pd.read_sql("SELECT * FROM regression_data", conn)

    # 获取 Total Pubs 和 Total Cites
    sql_pubs = "SELECT author_id, COUNT(pmid) as Total_Pubs FROM paper_authors GROUP BY author_id"
    df_pubs = pd.read_sql(sql_pubs, conn)

    sql_cites = """
    SELECT pa.author_id, SUM(IFNULL(pc.citation_count, 0)) as Total_Cites
    FROM paper_authors pa
    LEFT JOIN pmid_citations pc ON pa.pmid = pc.pmid
    GROUP BY pa.author_id
    """
    df_cites = pd.read_sql(sql_cites, conn)

    df_teams = pd.read_sql(f"SELECT scholar_id, Avg_Team_Size FROM scholar_avg_team_size_cache", conn)

    # 合并
    df = pd.merge(df_reg, df_pubs, left_on='scholar_id', right_on='author_id', how='inner')
    df = pd.merge(df, df_cites, on='author_id', how='inner')
    df = pd.merge(df, df_teams, on='scholar_id', how='inner')

    # 2. 变量转换与中心化
    df["Log_H_index"] = np.log1p(pd.to_numeric(df["H_index"], errors='coerce'))
    df["Log_Total_Pubs"] = np.log1p(pd.to_numeric(df["Total_Pubs"], errors='coerce'))
    df["Log_Total_Cites"] = np.log1p(pd.to_numeric(df["Total_Cites"], errors='coerce'))

    df["Log_Baseline_Pubs"] = np.log1p(pd.to_numeric(df["Baseline_Pubs"], errors='coerce'))
    df["Log_Team_Size"] = np.log1p(pd.to_numeric(df["Avg_Team_Size"], errors='coerce'))
    df["first_pub_year"] = pd.to_numeric(df["first_pub_year"], errors='coerce')

    # 中心化处理
    df["Year_C"] = df["first_pub_year"] - df["first_pub_year"].mean()
    df["Log_BasePubs_C"] = df["Log_Baseline_Pubs"] - df["Log_Baseline_Pubs"].mean()
    df["Log_Team_C"] = df["Log_Team_Size"] - df["Log_Team_Size"].mean()

    # 分类变量处理
    df["Cluster"] = df["Cluster"].astype(str)
    df["Cluster_Cat"] = pd.Categorical(df["Cluster"], categories=CLUSTER_ORDER, ordered=True)

    # 丢弃 NaN
    df = df.dropna(subset=['Log_H_index', 'Log_Total_Pubs', 'Log_Total_Cites', 'Year_C', 'Log_BasePubs_C', 'Log_Team_C', 'Cluster_Cat'])
    print(f"--> 数据准备就绪，最终样本量: {len(df)}")

    # 3. 运行 6 个回归模型
    print("Step 2: 正在拟合 6 个回归模型...")
    dvs = [
        ('Log_H_index', 'H-index'),
        ('Log_Total_Pubs', 'Total Pubs'),
        ('Log_Total_Cites', 'Total Cites')
    ]

    formula_unadj = f"~ C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))"
    formula_ctrl  = f"~ Year_C + Log_BasePubs_C + Log_Team_C + C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))"

    models = {}
    for dv_col, dv_name in dvs:
        models[f"{dv_name}_M1"] = smf.ols(formula=f"{dv_col} {formula_unadj}", data=df).fit()
        models[f"{dv_name}_M2"] = smf.ols(formula=f"{dv_col} {formula_ctrl}", data=df).fit()

    # 4. 格式化结果提取 (生成 CSV)
    print("Step 3: 格式化表格并导出 CSV...")

    def get_formatted_coef(mod, var_name):
        if var_name not in mod.params:
            return "-"
        coef = mod.params[var_name]
        se = mod.bse[var_name]
        pval = mod.pvalues[var_name]
        stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        return f"{coef:.3f}{stars} ({se:.4f})"

    var_mapping = {
        "Cluster 0 (Evolving T-Shaped)": f"C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))[T.0]",
        "Cluster 1 (Decaying Specialist)": f"C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))[T.1]",
        "Cluster 3 (Deepening Specialist)": f"C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))[T.3]",
        "Career Timing (Year Centered)": "Year_C",
        "Baseline Productivity (Log Centered)": "Log_BasePubs_C",
        "Average Team Size (Log Centered)": "Log_Team_C",
        "Constant": "Intercept"
    }

    # 构建 DataFrame 结构
    col_names = ['Variable', 'H-index M1 (Unadj)', 'H-index M2 (Ctrl)',
                 'Pubs M3 (Unadj)', 'Pubs M4 (Ctrl)',
                 'Cites M5 (Unadj)', 'Cites M6 (Ctrl)']

    rows = []

    # 标题 1
    rows.append(['Knowledge Evolution Trajectories (Ref: Cluster 2)'] + [''] * 6)
    for display_name, term_name in list(var_mapping.items())[:3]: # 前三个是 Clusters
        row = [display_name]
        for dv_col, dv_name in dvs:
            row.append(get_formatted_coef(models[f"{dv_name}_M1"], term_name))
            row.append(get_formatted_coef(models[f"{dv_name}_M2"], term_name))
        rows.append(row)

    # 标题 2
    rows.append(['Control Variables'] + [''] * 6)
    for display_name, term_name in list(var_mapping.items())[3:6]: # 中间三个是 Controls
        row = [display_name]
        for dv_col, dv_name in dvs:
            row.append(get_formatted_coef(models[f"{dv_name}_M1"], term_name))
            row.append(get_formatted_coef(models[f"{dv_name}_M2"], term_name))
        rows.append(row)

    # 常数项
    row = ['Constant']
    for dv_col, dv_name in dvs:
        row.append(get_formatted_coef(models[f"{dv_name}_M1"], "Intercept"))
        row.append(get_formatted_coef(models[f"{dv_name}_M2"], "Intercept"))
    rows.append(row)

    # R-squared 和 F-statistic
    r2_row = ['R-squared']
    f_row = ['F-statistic']
    for dv_col, dv_name in dvs:
        mod1, mod2 = models[f"{dv_name}_M1"], models[f"{dv_name}_M2"]
        r2_row.extend([f"{mod1.rsquared:.4f}", f"{mod2.rsquared:.4f}"])

        # F值也加星星
        f1_star = "***" if mod1.f_pvalue < 0.001 else ""
        f2_star = "***" if mod2.f_pvalue < 0.001 else ""
        f_row.extend([f"{mod1.fvalue:.2f}{f1_star}", f"{mod2.fvalue:.2f}{f2_star}"])

    rows.append(r2_row)
    rows.append(f_row)

    # 导出 CSV
    df_output = pd.DataFrame(rows, columns=col_names)
    out_path = os.path.join(OUTPUT_DIR, "Table_Regression_Results_Final.csv")
    df_output.to_csv(out_path, index=False, encoding='utf-8-sig')

    print("\n=== 表格预览 ===")
    print(df_output.to_string(index=False))
    print(f"\n=== CSV 已成功保存至: {out_path} ===")
    conn.close()

if __name__ == '__main__':
    prepare_and_run_hierarchical_regressions()

In [ ]:
"""Unified pipeline for Dual-Outcome Regression and Custom Pairwise Contrasts.

Design goals
------------
1. Run three OLS models: Log_H_index, Log_Total_Pubs, Log_Total_Cites.
2. Control for Year_C, Log_BasePubs_C, Log_Team_C.
3. Compute exact pairwise contrasts for ALL combinations.
4. Output a single, publication-ready CSV sorted exactly as requested:
   - Panel A: 0 vs 1, 0 vs 2, 0 vs 3
   - Panel B: 1 vs 2, 1 vs 3, 2 vs 3
"""

import sqlite3
import time
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Configuration
DB_PATH = r"F:\Pycharm_Temp\S.db"
OUTPUT_DIR = r"F:\Pycharm_Temp\True_Project\Appendix"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REFERENCE_CLUSTER = "1"
CLUSTER_ORDER = ["1", "0", "2", "3"]

CUSTOM_COMPARISON_ORDER = [
    ("0", "1"), ("0", "2"), ("0", "3"),  # Panel A
    ("1", "2"), ("1", "3"), ("2", "3")   # Panel B
]

# Helpers
def connect_db(db_path: str = DB_PATH) -> sqlite3.Connection:
    """创建数据库连接"""
    return sqlite3.connect(db_path)

# Data Preparation
def build_multi_outcome_frame(conn):
    print("Step 1: 构建包含三维绩效和控制变量的大宽表...")

    # 1. 基础回归表
    df_reg = pd.read_sql_query("SELECT * FROM regression_data", conn)

    # 2. 补齐 Total Pubs
    sql_pubs = "SELECT author_id as scholar_id, COUNT(pmid) as Total_Pubs FROM paper_authors GROUP BY author_id"
    df_pubs = pd.read_sql_query(sql_pubs, conn)

    # 3. 补齐 Total Cites
    sql_cites = """
    SELECT pa.author_id as scholar_id, SUM(IFNULL(pc.citation_count, 0)) as Total_Cites
    FROM paper_authors pa
    LEFT JOIN pmid_citations pc ON pa.pmid = pc.pmid
    GROUP BY pa.author_id
    """
    df_cites = pd.read_sql_query(sql_cites, conn)

    # 4. 获取 Team Size
    try:
        df_teams = pd.read_sql_query("SELECT scholar_id, Avg_Team_Size FROM scholar_avg_team_size_cache", conn)
    except Exception as e:
        raise Exception(f"Error: 找不到 scholar_avg_team_size_cache 表。错误信息: {e}")

    # --- 合并数据 ---
    df_final = pd.merge(df_reg, df_pubs, on='scholar_id', how='left')
    df_final = pd.merge(df_final, df_cites, on='scholar_id', how='left')
    df_final = pd.merge(df_final, df_teams, on='scholar_id', how='left')

    # 填补缺失
    df_final["Avg_Team_Size"] = df_final["Avg_Team_Size"].fillna(1.0)
    df_final = df_final.dropna(subset=["H_index", "Total_Pubs", "Total_Cites", "Baseline_Pubs", "first_pub_year", "Cluster"]).copy()

    # --- 变量转换 (Log + Centering) ---
    df_final["Log_H"] = np.log1p(pd.to_numeric(df_final["H_index"]))
    df_final["Log_Pubs"] = np.log1p(pd.to_numeric(df_final["Total_Pubs"]))
    df_final["Log_Cites"] = np.log1p(pd.to_numeric(df_final["Total_Cites"]))

    df_final["Log_BaseP"] = np.log1p(pd.to_numeric(df_final["Baseline_Pubs"]))
    df_final["Log_Team"] = np.log1p(pd.to_numeric(df_final["Avg_Team_Size"]))
    df_final["first_pub_year"] = pd.to_numeric(df_final["first_pub_year"])

    # 中心化
    df_final["Year_C"] = df_final["first_pub_year"] - df_final["first_pub_year"].mean()
    df_final["Log_BaseP_C"] = df_final["Log_BaseP"] - df_final["Log_BaseP"].mean()
    df_final["Log_Team_C"] = df_final["Log_Team"] - df_final["Log_Team"].mean()

    # 类别变量
    df_final["Cluster"] = df_final["Cluster"].astype(str)
    df_final["Cluster_Cat"] = pd.Categorical(df_final["Cluster"], categories=CLUSTER_ORDER, ordered=True)

    return df_final

# Pairwise Contrast Engine
def compute_custom_pairwise(model, dimension_name):
    """
    按照自定义顺序 (CUSTOM_COMPARISON_ORDER) 精确计算两两比较
    """
    params = model.params
    cov = model.cov_params()

    def term_name(c_label):
        if c_label == REFERENCE_CLUSTER: return None
        return f"C(Cluster_Cat, Treatment(reference='{REFERENCE_CLUSTER}'))[T.{c_label}]"

    rows = []

    for (a, b) in CUSTOM_COMPARISON_ORDER:
        # L 向量构造：比较 A 减 B (Effect of A over B)
        L = pd.Series(0.0, index=params.index)
        if a != REFERENCE_CLUSTER: L[term_name(a)] += 1.0
        if b != REFERENCE_CLUSTER: L[term_name(b)] -= 1.0

        # 估算值
        est = float(np.dot(L.values, params.reindex(L.index).values))
        # 方差与标准误
        cov_sub = cov.reindex(index=L.index, columns=L.index)
        var = float(np.dot(L.values, np.dot(cov_sub.values, L.values)))
        se = np.sqrt(var) if var >= 0 else np.nan

        # P 值
        z = est / se if se and se > 0 else np.nan
        p = 2 * (1 - stats.norm.cdf(abs(z))) if pd.notna(z) else np.nan

        # 置信区间
        ci_low = est - 1.96 * se if pd.notna(se) else np.nan
        ci_high = est + 1.96 * se if pd.notna(se) else np.nan

        # 转换为百分比效应量
        eff_pct = (np.exp(est) - 1) * 100
        ci_low_pct = (np.exp(ci_low) - 1) * 100 if pd.notna(ci_low) else np.nan
        ci_high_pct = (np.exp(ci_high) - 1) * 100 if pd.notna(ci_high) else np.nan

        # 格式化 P 值
        if p < 0.001: p_str = "<0.001 ***"
        elif p < 0.01: p_str = "<0.01 **"
        elif p < 0.05: p_str = "<0.05 *"
        else: p_str = f"{p:.3f}"

        # Panel 标记
        panel = "Panel A (vs Cluster 0)" if a == "0" else "Panel B (Among Others)"

        rows.append({
            "Panel": panel,
            "Comparison (A vs B)": f"{a} vs {b}",
            "Performance Dimension": dimension_name,
            "Log_Diff": round(est, 4),
            "SE": round(se, 4),
            "P-value": p_str,
            "Effect Size (%)": f"{eff_pct:+.1f}%",
            "95% CI": f"[{ci_low_pct:+.1f}%, {ci_high_pct:+.1f}%]"
        })

    return pd.DataFrame(rows)

# Main Execution
def run_all_dimensions():
    print("\n" + "="*70)
    print(" 🚀 开始执行三维绩效回归与定制化两两比较 🚀")
    print("="*70)
    t0 = time.time()

    conn = connect_db()
    try:
        df_final = build_multi_outcome_frame(conn)
    finally:
        conn.close() # 确保读取完数据后正确关闭连接

    # 待运行的三个模型
    target_models = [
        ("Log_H", "H-index"),
        ("Log_Pubs", "Total Publications"),
        ("Log_Cites", "Total Citations")
    ]

    base_formula = "~ Year_C + Log_BaseP_C + Log_Team_C + C(Cluster_Cat, Treatment(reference='1'))"

    all_results = pd.DataFrame()

    print("\nStep 2: 逐一拟合模型并计算对比...")
    for dv_col, dv_name in target_models:
        print(f"  --> 正在处理: {dv_name}")
        model = smf.ols(formula=f"{dv_col} {base_formula}", data=df_final).fit()

        # 提取定制对比
        df_contrast = compute_custom_pairwise(model, dimension_name=dv_name)
        all_results = pd.concat([all_results, df_contrast], ignore_index=True)


    dim_order = {"H-index": 1, "Total Publications": 2, "Total Citations": 3}
    all_results['Dim_Order'] = all_results['Performance Dimension'].map(dim_order)

    comp_order = {comp: idx for idx, comp in enumerate([f"{a} vs {b}" for a, b in CUSTOM_COMPARISON_ORDER])}
    all_results['Comp_Order'] = all_results['Comparison (A vs B)'].map(comp_order)

    all_results = all_results.sort_values(by=['Comp_Order', 'Dim_Order']).drop(columns=['Dim_Order', 'Comp_Order'])

    out_csv = os.path.join(OUTPUT_DIR, "Table_Custom_Pairwise_Comparisons.csv")
    all_results.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print("\n" + "="*90)
    print("🏆 最终两两比较结果预览 🏆")
    print("="*90)
    # 控制台预览只看核心列
    preview_cols = ["Panel", "Comparison (A vs B)", "Performance Dimension", "Effect Size (%)", "95% CI", "P-value"]
    print(all_results[preview_cols].to_string(index=False))
    print("="*90)

    print(f"\n✅ 完美生成！文件已保存至: {out_csv}")
    print(f"总耗时: {time.time() - t0:.1f}s")

if __name__ == "__main__":
    run_all_dimensions()